In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_119_Sirifort_Delhi_CPCB_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,197.25,302.14,21.84,78.44,44.95,16.88,20.39,0.69,6.76,...,0.12,NaN,10.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,205.44,323.81,22.39,118.26,81.11,23.93,21.13,0.68,7.02,...,0.16,NaN,10.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,203.83,313.44,22.11,103.84,69.57,28.11,21.24,0.90,7.05,...,0.17,NaN,10.00,NaN,0.20,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,230.18,351.16,22.55,91.05,65.79,29.63,21.05,0.75,9.17,...,0.12,NaN,10.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,185.58,308.63,22.35,78.91,59.49,24.19,21.57,0.81,20.42,...,0.15,NaN,10.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,191.63,243.57,23.01,26.90,49.87,43.64,47.23,2.12,22.01,...,0.16,14.18,96.23,0.98,105.48,0.0,0.0,7.52,NaN,NaN
362,2024-12-28,117.04,156.32,22.96,36.64,58.89,57.31,46.38,0.90,21.84,...,0.14,14.68,96.77,0.84,120.91,0.0,0.0,22.82,NaN,NaN
363,2024-12-29,125.68,160.04,23.01,33.59,56.60,56.90,47.19,0.62,21.78,...,0.57,13.69,91.25,2.66,154.64,0.0,0.0,77.03,NaN,NaN
364,2024-12-30,124.53,169.63,23.10,37.01,59.43,63.61,46.24,0.53,21.90,...,0.15,11.75,87.27,1.99,160.24,0.0,0.0,95.91,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 22)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Toluene (µg/m³)', 'Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
AT (°C)                0
RH (%)                 0
WS (m/s)               0
WD (deg)               0
RF (mm)                0
TOT-RF (mm)            0
SR (W/mt2)             0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         197.25        302.14       21.84        78.44   
1  2024-01-02         205.44        323.81       22.39        40.19   
2  2024-01-03         203.83        313.44       22.11       103.84   
3  2024-01-04         230.18        351.16       22.55        91.05   
4  2024-01-05         185.58        308.63       22.35        78.91   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      44.95        16.88        20.39        0.69           6.76   
1      81.11        23.93        21.13        0.68           7.02   
2      69.57        28.11        21.24        0.90           7.05   
3      65.79        29.63        21.05        0.75           9.17   
4      59.49        24.19        21.57        0.81          20.42   

   Benzene (µg/m³)  Eth-Benzene (µg/m³)  MP-Xylene (µg/m³)  AT (°C)  RH (%)  \
0             0.11                 0.03               0.

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2)
0,2024-01-01,2.200985,1.001249,-0.531826,2.302534,-1.087497,-1.774501,-1.413803,-0.690964,-1.937364,-0.964770,-1.372917,-1.097859,-0.063509,-1.667410,0.029711,0.055594,0.0,0.0,-0.074914
1,2024-01-02,2.357260,1.201484,-0.475551,0.004049,0.579134,-1.387835,-1.340349,-0.710691,-1.909763,-0.964770,0.380966,0.102489,-0.063509,-1.667410,0.029711,0.055594,0.0,0.0,-0.074914
2,2024-01-03,2.326539,1.105663,-0.504200,3.828849,0.047250,-1.158577,-1.329430,-0.276709,-1.906578,-0.858637,0.380966,0.402576,-0.063509,-1.667410,0.029711,0.055594,0.0,0.0,-0.074914
3,2024-01-04,2.829328,1.454204,-0.459179,3.060283,-0.126972,-1.075211,-1.348290,-0.572606,-1.681526,-1.070904,-1.372917,-1.097859,-0.063509,-1.667410,0.029711,0.055594,0.0,0.0,-0.074914
4,2024-01-05,1.978308,1.061218,-0.479643,2.330777,-0.417342,-1.373575,-1.296673,-0.454247,-0.487260,-1.070904,0.380966,-0.197598,-0.063509,-1.667410,0.029711,0.055594,0.0,0.0,-0.074914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,2.093749,0.460049,-0.412113,-0.794562,-0.860732,-0.306815,1.250405,2.129917,-0.318471,0.414964,0.380966,0.102489,-0.063509,1.523936,-0.195270,-0.692028,0.0,0.0,-2.082635
362,2024-12-28,0.670487,-0.346159,-0.417229,-0.209275,-0.444996,0.442933,1.166032,-0.276709,-0.336517,0.945631,0.380966,-0.497685,-0.063509,1.543921,-0.495244,-0.318096,0.0,0.0,-1.845704
363,2024-12-29,0.835348,-0.311785,-0.412113,-0.392553,-0.550543,0.420446,1.246434,-0.829049,-0.342887,0.521098,0.380966,0.402576,-0.063509,1.339628,0.029711,0.499321,0.0,0.0,-1.006224
364,2024-12-30,0.813404,-0.223172,-0.402904,-0.187041,-0.420107,0.788464,1.152135,-1.006587,-0.330148,0.627231,-1.372917,-0.197598,-0.063509,1.192329,1.968827,0.635032,0.0,0.0,-0.713854


In [10]:
df.to_excel('sirifort2024.xlsx', index=False)